In [1]:
!pip install pytesseract Pillow llama-index gradio transformers sentence-transformers
!apt-get update
!apt-get install -y tesseract-ocr

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [2]:
!pip install llama-index-embeddings-huggingface
import pytesseract
from PIL import Image
import gradio as gr
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, Document
from transformers import pipeline

# ✅ Set embedding model globally
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# ✅ Use a dictionary to hold state (avoids global scope issues in Colab)
state = {
    "documents": [],
    "index": None
}

qa_pipeline = pipeline("question-answering", model="distilbert-base-uncased-distilled-squad")

def ocr_and_index(image):
    # ✅ Fix for dark background images
    import numpy as np
    from PIL import ImageOps, ImageFilter

    image = image.convert("L")           # grayscale
    image = ImageOps.invert(image)       # invert dark→light background
    image = image.resize((image.width * 2, image.height * 2))  # upscale

    text = pytesseract.image_to_string(image)
    if not text.strip():
        return "⚠ No text found in image!"

    doc = Document(text=text)
    state["documents"] = []
    state["documents"].append(doc)
    state["index"] = None
    return f"✅ Text extracted!\n\n{text[:500]}"

def build_index():
    if not state["documents"]:
        return "⚠ No documents to index."
    state["index"] = VectorStoreIndex.from_documents(state["documents"])
    return f"✅ Index built! ({len(state['documents'])} document(s) indexed)"

def ask_question(question):
    if state["index"] is None:
        return "⚠ Please build the index first!"
    if not question.strip():
        return "⚠ Please type a question!"
    all_text = " ".join([doc.text for doc in state["documents"]])
    result = qa_pipeline(question=question, context=all_text)
    return result["answer"]

with gr.Blocks() as demo:
    gr.Markdown("## 📝 OCR + RAG QA System")

    with gr.Tab("Upload & Index"):
        image_input = gr.Image(type="pil", label="Upload Image")
        ocr_btn = gr.Button("Extract & Add to Index")
        ocr_output = gr.Textbox(label="Extracted Text Preview")
        build_btn = gr.Button("Build Index")
        build_output = gr.Textbox(label="Index Status")

        ocr_btn.click(ocr_and_index, inputs=image_input, outputs=ocr_output)
        build_btn.click(build_index, outputs=build_output)

    with gr.Tab("Ask Questions"):
        question_input = gr.Textbox(label="Your Question")
        ask_btn = gr.Button("Ask")
        answer_output = gr.Textbox(label="Answer")
        ask_btn.click(ask_question, inputs=question_input, outputs=answer_output)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f3cd2a91851657ce81.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
